# TP 4.1 — Analyse exploratoire (IVAL + jointure annuaire pour l’EP)

**Données** :

- IVAL GT : `donnees/fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv`
- Annuaire (priorité / EP) : `donnees/fr-en-annuaire-education.csv`
- Optionnel IPS : `donnees/fr-en-ips-lycees-ap2023.csv`

Voir `Docs/sources_data.md` section TP 4.1.

In [ ]:
from pathlib import Path

def resolve_data_dir() -> Path:
    """Racine du dépôt = dossier qui contient `donnees/`. Lancer Jupyter depuis cette racine."""
    cwd = Path.cwd()
    if (cwd / "donnees").is_dir():
        return cwd
    p = cwd
    for _ in range(4):
        if (p / "donnees").is_dir():
            return p
        p = p.parent
    return cwd

ROOT = resolve_data_dir()
DATA = ROOT / "donnees"
print("Répertoire données :", DATA.resolve())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


def load_depp_csv(path: Path, lowercase_cols: bool = False) -> pd.DataFrame:
    """CSV DEPP : séparateur ';', BOM UTF-8 possible sur la 1re colonne."""
    df = pd.read_csv(path, sep=";", low_memory=False, encoding="utf-8-sig")
    df.columns = df.columns.str.strip().str.lstrip("\ufeff")
    if lowercase_cols:
        df.columns = df.columns.str.lower()
    return df


ival_path = DATA / "fr-en-indicateurs-de-resultat-des-lycees-gt_v2.csv"
ann_path = DATA / "fr-en-annuaire-education.csv"
if not ival_path.exists() or not ann_path.exists():
    raise FileNotFoundError("Placez les CSV IVAL et Annuaire dans donnees/ (voir donnees/README.md)")

ival = load_depp_csv(ival_path)
# Annuaire : certains exports ont Identifiant_... (PascalCase), d'autres identifiant_...
ann = load_depp_csv(ann_path, lowercase_cols=True)

# Libellés bruts IVAL (accents, espaces) -> noms stables pour la suite du TP
IVAL_RENAME = {
    "Année": "annee",
    "UAI": "uai",
    "Secteur": "secteur",
    "Académie": "libelle_academie",
    "Taux de réussite - Toutes séries": "taux_reu_total",
    "Valeur ajoutée du taux de réussite - Toutes séries": "va_reu_total",
    "Taux d'accès 2nde-bac": "taux_acces_2nde",
    "Valeur ajoutée du taux d'acces 2nde-bac": "va_acces_2nde",
    "Taux de mentions - Toutes séries": "taux_men_total",
    "Valeur ajoutée du taux de mentions - Toutes séries": "va_men_total",
}
ival = ival.rename(columns={k: v for k, v in IVAL_RENAME.items() if k in ival.columns})
ival["uai"] = ival["uai"].astype(str).str.strip()

print("IVAL :", ival.shape, "| Annuaire :", ann.shape)
print("Colonnes annuaire (extrait) :", [c for c in ann.columns if "identifiant" in c or "prioritaire" in c])
ival.head(2)


In [ ]:
# Harmoniser types numériques (virgule décimale)
for c in ["taux_reu_total", "va_reu_total", "taux_acces_2nde", "taux_men_total"]:
    if c in ival.columns:
        ival[c] = pd.to_numeric(ival[c].astype(str).str.replace(",", "."), errors="coerce")

ival[["uai", "annee", "secteur", "libelle_academie", "taux_reu_total", "va_reu_total"]].describe(include="all")


In [ ]:
# Jointure avec l'annuaire pour disposer du flag EP (REP / REP+)
col_uai = "identifiant_de_l_etablissement"
col_ep = "appartenance_education_prioritaire"
missing = [c for c in (col_uai, col_ep) if c not in ann.columns]
if missing:
    raise KeyError(
        f"Colonnes absentes dans l'annuaire : {missing}. "
        f"Vérifiez le fichier et l'encodage (utf-8-sig). Colonnes disponibles : {list(ann.columns[:8])}…"
    )

sub = (
    ann[[col_uai, col_ep]]
    .rename(columns={col_uai: "uai", col_ep: "zone_ep_raw"})
    .assign(uai=lambda d: d["uai"].astype(str).str.strip())
    .drop_duplicates(subset=["uai"])
)

merged = ival.merge(sub, on="uai", how="left")
merged["zone_ep"] = merged["zone_ep_raw"].fillna("Hors EP").replace({"": "Hors EP", "nan": "Hors EP"})
merged["zone_ep"].value_counts(dropna=False).head(10)


In [ ]:
# Statistiques par académie, secteur, zone EP
g = (
    merged.groupby(["libelle_academie", "secteur", "zone_ep"], dropna=False)
    .agg(
        taux_reu_median=("taux_reu_total", "median"),
        taux_reu_mean=("taux_reu_total", "mean"),
        n=("uai", "count"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
g.head(12)


In [ ]:
# Histogramme des valeurs ajoutées (réussite)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(merged["va_reu_total"].dropna(), kde=True, ax=ax)
ax.set_title("Distribution de la valeur ajoutée (réussite) — IVAL GT")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot par académie (limiter aux académies les plus représentées pour lisibilité)
top_acad = merged["libelle_academie"].value_counts().head(8).index
subp = merged[merged["libelle_academie"].isin(top_acad)]
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=subp, x="libelle_academie", y="va_reu_total", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_title("Valeur ajoutée (réussite) par académie (échantillon des 8 académies les plus fournies)")
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap de corrélation entre quelques taux
cols = [c for c in ["taux_reu_total", "taux_acces_2nde", "taux_men_total", "va_reu_total", "va_acces_2nde", "va_men_total"] if c in merged.columns]
corr = merged[cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Corrélations (taux / VA)")
plt.tight_layout()
plt.show()


In [ ]:
# Comparaison public / privé sous contrat
fig, ax = plt.subplots(figsize=(6, 4))
sns.violinplot(data=merged, x="secteur", y="taux_reu_total", ax=ax)
ax.set_title("Taux brut de réussite par secteur")
plt.tight_layout()
plt.show()


## Optionnel — enrichissement IPS lycées

Si `donnees/fr-en-ips-lycees-ap2023.csv` est présent, on peut croiser l’indice de position sociale (voie GT).

In [ ]:
ips_path = DATA / "fr-en-ips-lycees-ap2023.csv"
if ips_path.exists():
    ips = load_depp_csv(ips_path)
    ips_sub = ips[["uai", "ips_voie_gt"]].rename(columns={"ips_voie_gt": "ips_gt"}).drop_duplicates("uai")
    ips_sub["uai"] = ips_sub["uai"].astype(str).str.strip()
    merged_ips = merged.merge(ips_sub, on="uai", how="left")
    print("Lignes avec IPS :", merged_ips["ips_gt"].notna().sum(), "/", len(merged_ips))
    merged_ips[["uai", "libelle_academie", "ips_gt", "va_reu_total"]].dropna(subset=["ips_gt"]).head()
else:
    print("IPS non téléchargé — voir donnees/README.md (optionnel)")

## Interprétation (10 lignes max)

1. …
2. …
3. …

**Limites** : effets de structure, seuils de publication DEPP, corrélation ≠ causalité, lecture prudente des comparaisons « valeur ajoutée ».